# Lab 01 — TF-IDF Representation and Document Search

Notebook này thực hiện các phần thí nghiệm, kiểm chứng, tìm kiếm, đánh giá và phân tích lỗi từ Part D đến Part J.

In [1]:
from pathlib import Path
import gc
import sys
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'labs').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
sys.path.insert(0, str(REPO_ROOT))

from labs.lab01.implementation import (
    build_representation, build_vocabulary, compute_counts, compute_idf,
    compute_tf, compute_tfidf, cosine_similarity,
    explain_similarity_contributions, heldout_oov_rate, inspect_terms,
    load_c4_documents, make_subword_tokenize, minimal_tokenize,
    normalized_tokenize, query_oov_rate, representation_summary,
    retrieval_metrics, search, train_subword_tokenizer,
)

DATA_PATH = REPO_ROOT / 'data/raw/c4-train.00000-of-01024-30K.json.gz'
RESULTS_PATH = REPO_ROOT / 'labs/lab01/results.csv'
assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH}'

## 7. Part D — Inspect the Sparse Representation

### 7.1. Dataset

Sử dụng nguyên corpus 30K documents do giảng viên cung cấp.

In [2]:
documents = load_c4_documents(DATA_PATH)
print(f'Number of documents: {len(documents):,}')
case_sensitive_vocabulary = {token for text in documents['text'] for token in text.split()}
lowercase_vocabulary = {token for text in documents['text'] for token in text.lower().split()}
print(f'Case-sensitive vocabulary: {len(case_sensitive_vocabulary):,}')
print(f'Lowercased vocabulary: {len(lowercase_vocabulary):,}')
print(f'Reduction from lowercasing: {len(case_sensitive_vocabulary) - len(lowercase_vocabulary):,}')
del case_sensitive_vocabulary, lowercase_vocabulary
documents.head(3)

Number of documents: 30,000


Case-sensitive vocabulary: 542,753
Lowercased vocabulary: 473,388
Reduction from lowercasing: 69,365


,document_id,text,url,timestamp
0,0,Beginners BBQ Class Taking Place in Missoula!\...,https://klyq.com/beginners-bbq-class-taking-pl...,2019-04-25T12:57:54Z
1,1,Discussion in 'Mac OS X Lion (10.7)' started b...,https://forums.macrumors.com/threads/restore-f...,2019-04-21T10:07:13Z
2,2,Foil plaid lycra and spandex shortall with met...,https://awishcometrue.com/Catalogs/Clearance/T...,2019-04-25T10:40:23Z


### 7.2. Xây dựng pipeline

Pipeline A thực hiện lowercase, tokenization theo khoảng trắng, CountVectorizer, TF, IDF và tạo TF-IDF matrix.

In [3]:
pipeline_a = build_representation(documents['text'], 'A — Minimal', minimal_tokenize)
summary_a = representation_summary(pipeline_a)
print('Pipeline A đã được xây dựng từ toàn bộ corpus.')

Pipeline A đã được xây dựng từ toàn bộ corpus.


### 7.3. Kiểm tra kích thước

Ghi lại `N`, `V` và matrix shape của biểu diễn Pipeline A.

In [4]:
number_of_documents, vocabulary_size = pipeline_a.counts.shape
pd.DataFrame([{
    'N — number of documents': number_of_documents,
    'V — vocabulary size': vocabulary_size,
    'matrix shape': f'{number_of_documents:,} × {vocabulary_size:,}',
}])

,N — number of documents,V — vocabulary size,matrix shape
0,30000,473388,"30,000 × 473,388"


### 7.4. Kiểm tra sparsity

Tính `1 - nnz(X)/(N × V)` và ghi lại số phần tử khác 0.

In [5]:
nonzero_entries = pipeline_a.counts.nnz
total_entries = number_of_documents * vocabulary_size
pd.DataFrame([{
    'nonzero entries': nonzero_entries,
    'total entries': total_entries,
    'sparsity': 1 - nonzero_entries / total_entries,
}])

,nonzero entries,total entries,sparsity
0,5519587,14201640000,0.999611


### 7.5. Inspect vocabulary

So sánh 20 terms phổ biến nhất theo document frequency, 20 terms có IDF cao nhất và 20 terms có TF-IDF cao nhất trong document 0.

In [6]:
top_df, top_idf, top_tfidf = inspect_terms(pipeline_a, document_id=0)
print('20 terms phổ biến nhất theo document frequency')
display(top_df)
print('20 terms có IDF cao nhất')
display(top_idf)
print('20 terms có TF-IDF cao nhất trong document 0')
display(top_tfidf)

20 terms phổ biến nhất theo document frequency


,term,document_frequency
0,the,27870
1,and,27385
2,to,26646
3,of,25999
4,a,25753
5,in,25042
6,for,23556
7,is,22673
8,with,21323
9,on,19936


20 terms có IDF cao nhất


,term,idf
0, ,10.308953
1,🤳:,10.308953
2,🤰:,10.308953
3,🤢🤢,10.308953
4,🤢.,10.308953
5,🤗🤗🤗,10.308953
6,🤖:,10.308953
7,🙏🏻,10.308953
8,🙌🏻👯❤,10.308953
9,🙌,10.308953


20 terms có TF-IDF cao nhất trong document 0


,term,tfidf
0,bbq,0.140189
1,class,0.106166
2,missoula!,0.079300
3,bbq?,0.079300
4,balay,0.079300
5,meat,0.078286
6,lonestar,0.073968
7,kcbs,0.073968
8,"timelines,",0.073968
9,"trimming,",0.063304


### 7.6. Nhận xét và trả lời câu hỏi Part D

Pipeline A tạo ma trận có 30.000 hàng và 473.388 cột. Ma trận có tổng cộng 5.519.587 phần tử khác 0, trong khi số ô lý thuyết lớn hơn rất nhiều. Sparsity bằng 0,999611, tức là phần lớn các ô đều bằng 0.

Mỗi tài liệu chỉ dùng một phần nhỏ vocabulary, nhưng mọi vector vẫn phải có cùng số chiều. Nhờ đó, một vị trí trong vector luôn đại diện cho cùng một từ và các tài liệu mới có thể so sánh với nhau.

Các từ xuất hiện trong nhiều tài liệu nhất là `the`, `and`, `to`, `of`. Đây là các từ rất phổ biến nên IDF thấp. Ngược lại, danh sách các từ có IDF cao nhất có nhiều emoji, ký hiệu và chuỗi ký tự lỗi. Kết quả này cho thấy từ hiếm chưa chắc là từ có ích.

Trong tài liệu đầu tiên, các từ như `bbq` và `class` có TF-IDF cao vì chúng xuất hiện trong tài liệu đó và không quá phổ biến trong corpus. Một từ có IDF cao chỉ có TF-IDF cao khi từ đó thực sự xuất hiện trong tài liệu.

## 8. Part E — Core Implementation

### 8.1. Mục tiêu

Tự triển khai phiên bản TF-IDF tối giản mà không gọi trực tiếp `TfidfVectorizer`. Các hàm được viết trong `implementation.py` và được kiểm thử độc lập ở các mục dưới đây.


### 8.2. Các hàm cần xây dựng

Sáu hàm cốt lõi gồm `build_vocabulary`, `compute_counts`, `compute_tf`, `compute_idf`, `compute_tfidf` và `cosine_similarity`.


In [7]:
core_functions = {
    'build_vocabulary': build_vocabulary,
    'compute_counts': compute_counts,
    'compute_tf': compute_tf,
    'compute_idf': compute_idf,
    'compute_tfidf': compute_tfidf,
    'cosine_similarity': cosine_similarity,
}
pd.DataFrame({
    'function': core_functions.keys(),
    'callable': [callable(function) for function in core_functions.values()],
})


,function,callable
0,build_vocabulary,True
1,compute_counts,True
2,compute_tf,True
3,compute_idf,True
4,compute_tfidf,True
5,cosine_similarity,True


### 8.3. Corpus kiểm thử

Corpus nhỏ gồm ba tài liệu; vocabulary kỳ vọng được sắp xếp theo alphabet để kết quả có thể kiểm tra lặp lại.


In [8]:
toy_corpus = ['cat eats fish', 'dog eats fish', 'cat likes fish']
expected_vocabulary = {'cat': 0, 'dog': 1, 'eats': 2, 'fish': 3, 'likes': 4}
pd.DataFrame({'document': toy_corpus})


,document
0,cat eats fish
1,dog eats fish
2,cat likes fish


### 8.4. Unit tests

Mỗi hàm cốt lõi có ít nhất một phép kiểm tra bằng `assert`. Cell chỉ hoàn thành khi cả sáu phép kiểm tra đều đúng.


In [9]:
vocabulary = build_vocabulary(toy_corpus)
assert vocabulary == expected_vocabulary

count_matrix = [compute_counts(document, vocabulary) for document in toy_corpus]
assert count_matrix[0] == [1, 0, 1, 1, 0]

tf_d1 = compute_tf(count_matrix[0])
assert np.allclose(tf_d1, [1/3, 0, 1/3, 1/3, 0])

idf_values = compute_idf(count_matrix)
assert np.allclose(idf_values, [np.log(3/2), np.log(3), np.log(3/2), 0, np.log(3)])

tfidf_d1 = compute_tfidf(tf_d1, idf_values)
assert np.allclose(tfidf_d1, [np.log(3/2)/3, 0, np.log(3/2)/3, 0, 0])

similarity = cosine_similarity([1, 1, 1], [1, 1, 0])
assert np.isclose(similarity, 2 / np.sqrt(6))

print('All six core functions passed their unit checks.')


All six core functions passed their unit checks.


### 8.5. So sánh với thư viện

Scikit-learn mặc định dùng IDF có cộng 1, còn phần tự cài đặt dùng `ln(N/DF)`. Vì vậy, kết quả tham chiếu được đưa về cùng convention trước khi so sánh.


In [10]:
reference_counts = CountVectorizer(vocabulary=vocabulary).transform(toy_corpus)
reference = TfidfTransformer(
    norm=None, use_idf=True, smooth_idf=False, sublinear_tf=False
).fit_transform(reference_counts).toarray()
row_totals = np.asarray(reference_counts.sum(axis=1)).ravel()[:, None]
adjusted_reference = reference / row_totals - reference_counts.toarray() / row_totals
student_tfidf = np.asarray([
    compute_tfidf(compute_tf(row), idf_values) for row in count_matrix
])
assert np.allclose(student_tfidf, adjusted_reference)

comparison = pd.DataFrame({
    'term': list(vocabulary),
    'manual_document_1': student_tfidf[0],
    'sklearn_adjusted_document_1': adjusted_reference[0],
})
print('Student TF-IDF matches the adjusted scikit-learn reference.')
comparison


Student TF-IDF matches the adjusted scikit-learn reference.


,term,manual_document_1,sklearn_adjusted_document_1
0,cat,0.135155,0.135155
1,dog,0.000000,0.000000
2,eats,0.135155,0.135155
3,fish,0.000000,0.000000
4,likes,0.000000,0.000000


## 9. Part F — Preprocessing Ablation

### 9.1. Pipeline A — Minimal

Lowercase và tokenization theo khoảng trắng. Cell dưới đây minh họa trực tiếp đầu ra tokenizer trên tài liệu đầu tiên.


In [11]:
sample_text = documents['text'].iloc[0]
print('Pipeline A — 30 tokens đầu tiên:')
print(minimal_tokenize(sample_text)[:30])


Pipeline A — 30 tokens đầu tiên:
['beginners', 'bbq', 'class', 'taking', 'place', 'in', 'missoula!', 'do', 'you', 'want', 'to', 'get', 'better', 'at', 'making', 'delicious', 'bbq?', 'you', 'will', 'have', 'the', 'opportunity,', 'put', 'this', 'on', 'your', 'calendar', 'now.', 'thursday,', 'september']


### 9.2. Pipeline B — Normalized

Lowercase, chuẩn hóa punctuation, tokenization và loại stopword. Đầu ra dưới đây dùng cùng tài liệu với Pipeline A để dễ đối chiếu.


In [12]:
print('Pipeline B — 30 tokens đầu tiên:')
print(normalized_tokenize(sample_text)[:30])


Pipeline B — 30 tokens đầu tiên:
['beginners', 'bbq', 'class', 'taking', 'place', 'missoula', 'want', 'better', 'making', 'delicious', 'bbq', 'opportunity', 'calendar', 'thursday', 'september', '22nd', 'join', 'world', 'class', 'bbq', 'champion', 'tony', 'balay', 'lonestar', 'smoke', 'rangers', 'teaching', 'beginner', 'level', 'class']


### 9.3. Pipeline C — Extended

Pipeline C chuẩn hóa văn bản rồi dùng subword tokenization BPE. Mô hình BPE chỉ được huấn luyện trên 90% đầu của corpus; 10% còn lại được giữ riêng để đo OOV.


In [13]:
split_index = int(len(documents) * 0.9)
training_texts = documents['text'].iloc[:split_index].tolist()
heldout_texts = documents['text'].iloc[split_index:].tolist()
subword_model = train_subword_tokenizer(training_texts)
subword_tokenize = make_subword_tokenize(subword_model)

pipelines = [
    ('A — Minimal', minimal_tokenize),
    ('B — Normalized', normalized_tokenize),
    ('C — Extended BPE', subword_tokenize),
]

print('Pipeline C — 30 subword tokens đầu tiên:')
print(subword_tokenize(sample_text)[:30])


Pipeline C — 30 subword tokens đầu tiên:
['beginners', 'bbq', 'class', 'taking', 'place', 'in', 'miss', 'ou', 'la', '!', 'do', 'you', 'want', 'to', 'get', 'better', 'at', 'making', 'delicious', 'bbq', '?', 'you', 'will', 'have', 'the', 'opportunity', ',', 'put', 'this', 'on']


### 9.4. Thiết lập và kết quả so sánh

Ba pipeline được so sánh theo vocabulary size, số token trung bình mỗi document, sparsity, held-out OOV, query OOV và search performance. Tám query cố định dưới đây được dùng thống nhất cho cả ba pipeline.


In [14]:
queries = [
    'medical image classification',
    'transformer language model',
    'deep learning healthcare',
    'natural language processing',
    'climate change policy',
    'software development tools',
    'financial market analysis',
    'online education courses',
]
pd.DataFrame({'query': queries})


,query
0,medical image classification
1,transformer language model
2,deep learning healthcare
3,natural language processing
4,climate change policy
5,software development tools
6,financial market analysis
7,online education courses


In [15]:
summaries = []
retrieval_rows = []
pipeline_b = None

for pipeline_name, tokenizer in pipelines:
    model = pipeline_a if pipeline_name == 'A — Minimal' else build_representation(
        documents['text'], pipeline_name, tokenizer
    )
    if pipeline_name == 'B — Normalized':
        pipeline_b = model
    summary = representation_summary(model)
    known_vocabulary = set(subword_model.get_vocab()) if pipeline_name == 'C — Extended BPE' else None
    summary['heldout_oov_rate'] = heldout_oov_rate(
        training_texts, heldout_texts, tokenizer, known_vocabulary
    )
    summary['query_oov_rate'] = query_oov_rate(model, queries)
    summaries.append(summary)

    for query in queries:
        ranked = search(query, model, documents, top_k=5)
        ranked.insert(0, 'query', query)
        ranked.insert(0, 'pipeline', pipeline_name)
        retrieval_rows.append(ranked)

    if model is not pipeline_a and model is not pipeline_b:
        del model
        gc.collect()

ablation_results = pd.DataFrame(summaries)
retrieval_results = pd.concat(retrieval_rows, ignore_index=True)
ablation_results


,pipeline,documents,vocabulary_size,average_tokens_per_document,nonzero_entries,sparsity,heldout_oov_rate,query_oov_rate
0,A — Minimal,30000,473388,361.089067,5519587,0.999611,0.039791,0.0
1,B — Normalized,30000,191714,196.737367,3755479,0.999347,0.033674,0.0
2,C — Extended BPE,30000,29732,451.706733,5665986,0.993648,0.000123,0.0


### 9.5. Câu hỏi phân tích

1. Lowercasing làm vocabulary giảm từ 542.753 xuống 473.388 từ. Các dạng như `The` và `the` được gộp thành một từ nên số lượng từ khác nhau giảm 69.365.

2. Stopword removal không phải lúc nào cũng cải thiện representation. Trong thí nghiệm này Pipeline B cho search performance tốt nhất, nhưng một số stopword vẫn có thể mang thông tin tùy query và nhiệm vụ.

3. Dấu câu có thể mang thông tin về ranh giới câu, từ ghép, chữ viết tắt, số thập phân, URL hoặc địa chỉ email; loại toàn bộ punctuation có thể làm mất các dấu hiệu này.

4. Pipeline A tạo sparse matrix nhất vì vocabulary lớn nhưng mỗi document chỉ chứa một số ít token. Pipeline C có vocabulary nhỏ hơn nên tỷ lệ ô khác 0 cao hơn.

5. Trên candidate pool và các nhãn relevance hiện có, Pipeline B cho search tốt nhất với P@5 = 0,55, Recall@5 = 0,9375 và MRR = 0,6875. Pipeline C có cùng P@5 và Recall@5 nhưng MRR thấp hơn một chút, bằng 0,675.

6. Vocabulary nhỏ hơn cũng không tự động làm kết quả tìm kiếm tốt hơn. Pipeline C có vocabulary nhỏ nhất nhưng không vượt Pipeline B.

## 10. Part G — Application: Document Search

### 10.1. Bài toán

Nhận một user query và trả về Top-K documents từ corpus 30K.


### 10.2. Pipeline tìm kiếm

Query được biến đổi sang TF-IDF theo cùng vocabulary và IDF của từng pipeline, sau đó cosine similarity được dùng để xếp hạng.


### 10.3. Query examples

Tám query đã khai báo ở Part F được dùng thống nhất cho cả ba pipeline. Bảng query ở mục 9.4 là đầu vào đầy đủ của thí nghiệm tìm kiếm.


### 10.4. Kết quả Top-5

Mỗi bảng dưới đây hiển thị pipeline, rank, document ID, similarity và document preview cho một query.


In [16]:
for query in queries:
    print(f'\nQUERY: {query}')
    display(retrieval_results[retrieval_results['query'] == query])


QUERY: medical image classification


,pipeline,query,rank,document_id,similarity,document_preview
0,A — Minimal,medical image classification,1,18971,0.428478,The new RTS Environmental Classification syste...
1,A — Minimal,medical image classification,2,8370,0.215330,What is a Online Medical Second Opinion? For o...
2,A — Minimal,medical image classification,3,190,0.215071,This title is a comprehensive account of the k...
3,A — Minimal,medical image classification,4,12982,0.208964,"Sad moments, those when we feel bad, can be cl..."
4,A — Minimal,medical image classification,5,27781,0.204072,❶Press Officer Resume Sample. Based on your re...
40,B — Normalized,medical image classification,1,18971,0.447555,The new RTS Environmental Classification syste...
41,B — Normalized,medical image classification,2,8527,0.367814,History of maize classification. How races use...
42,B — Normalized,medical image classification,3,19908,0.235452,Download League Of Legends Wallpapers in high-...
43,B — Normalized,medical image classification,4,17794,0.230471,Filters the output of 'wp_calculate_image_size...
44,B — Normalized,medical image classification,5,12658,0.228429,"This guidance is for pharmacists who handle, u..."



QUERY: transformer language model


,pipeline,query,rank,document_id,similarity,document_preview
5,A — Minimal,transformer language model,1,27936,0.290718,"hi, I am having problems with transformer / ci..."
6,A — Minimal,transformer language model,2,25428,0.218338,"Note: If you're on an iPhone, you cannot chang..."
7,A — Minimal,transformer language model,3,24482,0.214497,"Harald, you are a co-owner of Language Partner..."
8,A — Minimal,transformer language model,4,701,0.207334,Program in Teaching French as a Foreign Langua...
9,A — Minimal,transformer language model,5,4075,0.194890,Looking for Spanish language instructor to imp...
45,B — Normalized,transformer language model,1,27936,0.505846,"hi, I am having problems with transformer / ci..."
46,B — Normalized,transformer language model,2,25428,0.275951,"Note: If you're on an iPhone, you cannot chang..."
47,B — Normalized,transformer language model,3,24482,0.219072,"Harald, you are a co-owner of Language Partner..."
48,B — Normalized,transformer language model,4,701,0.204292,Program in Teaching French as a Foreign Langua...
49,B — Normalized,transformer language model,5,4289,0.200185,"Commercial Building Properties, Commercial Bui..."



QUERY: deep learning healthcare


...

,pipeline,query,rank,document_id,similarity,document_preview
25,A — Minimal,software development tools,1,2856,0.517805,The document Software Development Basics gives...
26,A — Minimal,software development tools,2,7309,0.291334,With the improvement of digital cameras as wel...
27,A — Minimal,software development tools,3,21363,0.282961,The below list is a selected “terminology” for...
28,A — Minimal,software development tools,4,12763,0.253812,Why Clinic Management Software is Must for Cli...
29,A — Minimal,software development tools,5,10105,0.251102,Paul Berg is an independent consultant special...
65,B — Normalized,software development tools,1,2856,0.529788,The document Software Development Basics gives...
66,B — Normalized,software development tools,2,7309,0.342260,With the improvement of digital cameras as wel...
67,B — Normalized,software development tools,3,21363,0.315177,The below list is a selected “terminology” for...
68,B — Normalized,software development tools,4,22190,0.306317,"Brains, behaviour, statistics, and random thou..."
69,B — Normalized,software development tools,5,17461,0.284457,"Via the Big Slash, comes ONLamp.com: Calculati..."



QUERY: financial market analysis


,pipeline,query,rank,document_id,similarity,document_preview
30,A — Minimal,financial market analysis,1,7607,0.264428,Select a commercial project in manufacturing a...
31,A — Minimal,financial market analysis,2,22926,0.262383,Belay Devices Market Report 2018-2023 has been...
32,A — Minimal,financial market analysis,3,21404,0.253498,Keep in mind the historical returns of differe...
33,A — Minimal,financial market analysis,4,28964,0.251391,The global “Network Cables market” research re...
34,A — Minimal,financial market analysis,5,5249,0.243011,Focusing only on shareholders’ financial retur...
70,B — Normalized,financial market analysis,1,22926,0.379183,Belay Devices Market Report 2018-2023 has been...
71,B — Normalized,financial market analysis,2,28964,0.306697,The global “Network Cables market” research re...
72,B — Normalized,financial market analysis,3,5249,0.293863,Focusing only on shareholders’ financial retur...
73,B — Normalized,financial market analysis,4,21404,0.292624,Keep in mind the historical returns of differe...
74,B — Normalized,financial market analysis,5,22246,0.289099,The Comprehensive Annual Financial Report (CAF...



QUERY: online education courses


,pipeline,query,rank,document_id,similarity,document_preview
35,A — Minimal,online education courses,1,12474,0.356181,Get complete access to all courses on this sit...
36,A — Minimal,online education courses,2,23045,0.324300,I'm going to turn you into a course creation e...
37,A — Minimal,online education courses,3,28891,0.288454,Spice up your online education and take the De...
38,A — Minimal,online education courses,4,21561,0.282661,The York College Writing Across the Curriculum...
39,A — Minimal,online education courses,5,6123,0.246447,"With today’s advancement in technology, it is ..."
75,B — Normalized,online education courses,1,23045,0.477547,I'm going to turn you into a course creation e...
76,B — Normalized,online education courses,2,28891,0.452010,Spice up your online education and take the De...
77,B — Normalized,online education courses,3,12474,0.399713,Get complete access to all courses on this sit...
78,B — Normalized,online education courses,4,7416,0.395576,Download a different browser for free! Welcome...
79,B — Normalized,online education courses,5,21561,0.351673,The York College Writing Across the Curriculum...


### Nhận xét Part G

Các bảng Top-5 cho thấy preprocessing làm thay đổi similarity và thứ tự xếp hạng. Chất lượng các kết quả này được lượng hóa bằng P@5, Recall@5 và MRR trong Part H.

## 11. Part H — Evaluation

### 11.1. Tạo evaluation set

Relevance labels được gán trên candidate pool là hợp của các kết quả Top-5 từ ba pipeline. Các nhãn vẫn cần được kiểm tra cuối cùng trước khi nộp.

In [17]:
# Nhãn được rà theo toàn văn trong candidate pool; vẫn cần người nộp kiểm tra cuối.
reviewed_relevant = {
    'climate change policy': {2231, 9209, 11682, 21770},
    'deep learning healthcare': set(),
    'financial market analysis': {21404},
    'medical image classification': set(),
    'natural language processing': set(),
    'online education courses': {6123, 7416, 23045, 28891},
    'software development tools': {2856, 21363, 22190},
    'transformer language model': set(),
}

label_template = retrieval_results[[
    'query', 'document_id', 'document_preview'
]].drop_duplicates().sort_values(['query', 'document_id'])
label_template['relevant'] = [
    int(document_id in reviewed_relevant.get(query, set()))
    for query, document_id in zip(label_template['query'], label_template['document_id'])
]
label_template['label_source'] = 'content reviewed; final human review pending'
judgments = label_template.set_index(['query', 'document_id'])['relevant']
retrieval_results['relevant'] = [
    int(judgments.loc[(query, document_id)])
    for query, document_id in zip(retrieval_results['query'], retrieval_results['document_id'])
]
retrieval_results['label_source'] = 'content reviewed; final human review pending'
retrieval_results.to_csv(RESULTS_PATH, index=False, encoding='utf-8')
print(f'Wrote retrieval results to {RESULTS_PATH}')
print('Relevance labels are kept in this notebook and included in results.csv.')
label_template.head()

Wrote retrieval results to D:\Project\NLP-2025\NLP-Labs\labs\lab01\results.csv
Relevance labels are kept in this notebook and included in results.csv.


,query,document_id,document_preview,relevant,label_source
21,climate change policy,2231,The Rupa Lake Cooperative in Nepal. Credit: Bi...,1,content reviewed; final human review pending
22,climate change policy,9209,Climate Change Denial: Why It can Be Hugely In...,1,content reviewed; final human review pending
23,climate change policy,11682,The existential threats to people and the envi...,1,content reviewed; final human review pending
20,climate change policy,19142,Reduce short-lived climate change or even this...,0,content reviewed; final human review pending
24,climate change policy,21770,See this article detailing that arctic sea ice...,1,content reviewed; final human review pending


In [18]:
# Metrics use the current candidate-pool labels.
labels = label_template.copy()
assert not labels['relevant'].isna().any(), 'Relevance labels chưa được gán đầy đủ.'

relevant_by_query = {
    query: set(group.loc[group['relevant'] == 1, 'document_id'].astype(int))
    for query, group in labels.groupby('query')
}
excluded_queries = [query for query, ids in relevant_by_query.items() if not ids]
print('Diagnostic queries excluded from aggregate metrics (no relevant item in pool):')
print(excluded_queries)

metric_rows = []
for (pipeline, query), group in retrieval_results.groupby(['pipeline', 'query'], sort=False):
    if not relevant_by_query.get(query, set()):
        continue
    ordered = group.sort_values('rank')['document_id'].astype(int).tolist()
    metrics = retrieval_metrics(ordered, relevant_by_query[query], k=5)
    metric_rows.append({'pipeline': pipeline, 'query': query, **metrics})

metrics_df = pd.DataFrame(metric_rows)
retrieval_results = retrieval_results.drop(
    columns=['precision@5', 'recall@5', 'reciprocal_rank'], errors='ignore'
).merge(metrics_df, on=['pipeline', 'query'], how='left')
retrieval_results.to_csv(RESULTS_PATH, index=False, encoding='utf-8')


Diagnostic queries excluded from aggregate metrics (no relevant item in pool):
['deep learning healthcare', 'medical image classification', 'natural language processing', 'transformer language model']


### 11.2. Precision@5

`P@5` là tỷ lệ document relevant trong năm kết quả đầu tiên. Bảng dưới đây cho biết Precision@5 của từng query được dùng trong evaluation.


In [19]:
metrics_df[['pipeline', 'query', 'precision@5']]


,pipeline,query,precision@5
0,A — Minimal,climate change policy,0.8
1,A — Minimal,software development tools,0.4
2,A — Minimal,financial market analysis,0.2
3,A — Minimal,online education courses,0.6
4,B — Normalized,climate change policy,0.8
5,B — Normalized,software development tools,0.6
6,B — Normalized,financial market analysis,0.2
7,B — Normalized,online education courses,0.6
8,C — Extended BPE,climate change policy,0.8
9,C — Extended BPE,software development tools,0.6


### 11.3. Recall@5

`Recall@5` là tỷ lệ document relevant trong candidate pool được tìm thấy ở Top-5.


In [20]:
metrics_df[['pipeline', 'query', 'recall@5']]


,pipeline,query,recall@5
0,A — Minimal,climate change policy,1.000000
1,A — Minimal,software development tools,0.666667
2,A — Minimal,financial market analysis,1.000000
3,A — Minimal,online education courses,0.750000
4,B — Normalized,climate change policy,1.000000
5,B — Normalized,software development tools,1.000000
6,B — Normalized,financial market analysis,1.000000
7,B — Normalized,online education courses,0.750000
8,C — Extended BPE,climate change policy,1.000000
9,C — Extended BPE,software development tools,1.000000


### 11.4. Mean Reciprocal Rank

Reciprocal rank của mỗi query là nghịch đảo hạng của document relevant đầu tiên. MRR là trung bình trên các query có ít nhất một document relevant trong candidate pool.


In [21]:
display(metrics_df[['pipeline', 'query', 'reciprocal_rank']])
print('Aggregate metrics by pipeline')
display(metrics_df.groupby('pipeline').mean(numeric_only=True))
print('Evidence for selecting strong and weak queries')
display(metrics_df.groupby('query').mean(numeric_only=True).sort_values('precision@5'))


,pipeline,query,reciprocal_rank
0,A — Minimal,climate change policy,0.500000
1,A — Minimal,software development tools,1.000000
2,A — Minimal,financial market analysis,0.333333
3,A — Minimal,online education courses,0.500000
4,B — Normalized,climate change policy,0.500000
5,B — Normalized,software development tools,1.000000
6,B — Normalized,financial market analysis,0.250000
7,B — Normalized,online education courses,1.000000
8,C — Extended BPE,climate change policy,0.500000
9,C — Extended BPE,software development tools,1.000000


Aggregate metrics by pipeline


,precision@5,recall@5,reciprocal_rank
pipeline,,,
A — Minimal,0.50,0.854167,0.583333
B — Normalized,0.55,0.937500,0.687500
C — Extended BPE,0.55,0.937500,0.675000


Evidence for selecting strong and weak queries


,precision@5,recall@5,reciprocal_rank
query,,,
financial market analysis,0.200000,1.000000,0.261111
software development tools,0.533333,0.888889,1.000000
online education courses,0.600000,0.750000,0.833333
climate change policy,0.800000,1.000000,0.500000


## 12. Part I — Error Analysis

### 12.1. Dữ liệu kiểm chứng

Chọn hai query có kết quả tốt và hai query chẩn đoán lỗi. Với Pipeline B, cell dưới đây hiển thị relevant IDs trong candidate pool, Top-5 IDs, relevant IDs bị bỏ sót, similarity và các từ đóng góp nhiều nhất.

In [22]:
analysis_queries = [
    'climate change policy',
    'online education courses',
    'medical image classification',
    'transformer language model',
]
analysis_summary_rows = []
analysis_detail_rows = []

for query in analysis_queries:
    relevant_ids = sorted(relevant_by_query.get(query, set()))
    ranked = retrieval_results[
        (retrieval_results['pipeline'] == 'B — Normalized')
        & (retrieval_results['query'] == query)
    ].sort_values('rank')
    returned_ids = ranked['document_id'].astype(int).tolist()
    missed_ids = sorted(set(relevant_ids) - set(returned_ids))
    analysis_summary_rows.append({
        'query': query,
        'relevant_ids_in_candidate_pool': relevant_ids,
        'returned_top5_ids': returned_ids,
        'missed_relevant_ids': missed_ids,
    })
    for row in ranked.itertuples(index=False):
        terms = explain_similarity_contributions(
            query, int(row.document_id), pipeline_b, max_terms=5
        )
        top_terms = ', '.join(
            f"{item.term} ({item.contribution:.4f})"
            for item in terms.itertuples(index=False)
        )
        analysis_detail_rows.append({
            'query': query,
            'rank': int(row.rank),
            'document_id': int(row.document_id),
            'relevant': int(row.document_id in relevant_ids),
            'similarity': float(row.similarity),
            'top_contributing_terms': top_terms,
        })

analysis_summary_df = pd.DataFrame(analysis_summary_rows)
analysis_details_df = pd.DataFrame(analysis_detail_rows)
print('Pipeline B — candidate-pool relevance and Top-5 coverage')
display(analysis_summary_df)
print('Pipeline B — similarity and normalized TF-IDF term contributions')
with pd.option_context('display.max_colwidth', None):
    display(analysis_details_df)

Pipeline B — candidate-pool relevance and Top-5 coverage


,query,relevant_ids_in_candidate_pool,returned_top5_ids,missed_relevant_ids
0,climate change policy,"[2231, 9209, 11682, 21770]","[19142, 2231, 9209, 11682, 21770]",[]
1,online education courses,"[6123, 7416, 23045, 28891]","[23045, 28891, 12474, 7416, 21561]",[6123]
2,medical image classification,[],"[18971, 8527, 19908, 17794, 12658]",[]
3,transformer language model,[],"[27936, 25428, 24482, 701, 4289]",[]


Pipeline B — similarity and normalized TF-IDF term contributions


,query,rank,document_id,relevant,similarity,top_contributing_terms
0,climate change policy,1,19142,0,0.641831,"climate (0.4867), change (0.1551)"
1,climate change policy,2,2231,1,0.543263,"climate (0.3694), change (0.0942), policy (0.0797)"
2,climate change policy,3,9209,1,0.483173,"climate (0.3698), change (0.1133)"
3,climate change policy,4,11682,1,0.447795,"climate (0.3657), change (0.0466), policy (0.0355)"
4,climate change policy,5,21770,1,0.440481,"climate (0.3644), change (0.0761)"
5,online education courses,1,23045,1,0.477547,courses (0.4775)
6,online education courses,2,28891,1,0.452010,"courses (0.2739), online (0.0966), education (0.0815)"
7,online education courses,3,12474,0,0.399713,courses (0.3997)
8,online education courses,4,7416,1,0.395576,"courses (0.2684), education (0.0799), online (0.0473)"
9,online education courses,5,21561,0,0.351673,"courses (0.3119), education (0.0398)"


### 12.2. Phân tích bốn query

Các bảng ngay phía trên dùng Pipeline B và các nhãn hiện có trong candidate pool. Giá trị trong ngoặc sau mỗi từ là tích giữa trọng số TF-IDF đã chuẩn hóa của query và document.

#### 12.2.1. Hai query có tài liệu relevant

**`climate change policy`** có bốn document relevant: 2231, 9209, 11682 và 21770. Cả bốn đều xuất hiện trong Top-5 ở các hạng 2–5, nên Recall@5 bằng 1,0 và P@5 bằng 0,8. Document 19142 đứng hạng 1 nhưng không relevant; điểm 0,6418 chủ yếu đến từ `climate` (0,4867) và `change` (0,1551), dù nội dung không trình bày chính sách khí hậu mạch lạc. Trong các kết quả relevant, `climate` và `change` đóng góp chính; `policy` cũng đóng góp ở document 2231 và 11682.

**`online education courses`** có bốn document relevant: 6123, 7416, 23045 và 28891. Top-5 trả về 23045, 28891, 12474, 7416 và 21561, do đó bỏ sót document 6123; P@5 bằng 0,6 và Recall@5 bằng 0,75. Document 12474 chỉ khớp `courses` (0,3997), còn document 21561 khớp `courses` (0,3119) và `education` (0,0398) nhưng không nói về học trực tuyến.

#### 12.2.2. Hai query chẩn đoán lỗi

**`medical image classification`** không có document relevant trong candidate pool. Document 18971 đứng đầu với điểm 0,4476 hoàn toàn do `classification`, nhưng nội dung là phân loại môi trường cho công trình. Document 8527 cũng chỉ khớp `classification` (0,3678) và nói về giống ngô; các kết quả còn lại lần lượt chỉ khớp `image` hoặc `medical`. Query này dùng để minh họa lỗi tìm kiếm và không được đưa vào metric tổng hợp.

**`transformer language model`** cũng không có document relevant trong candidate pool và không được đưa vào metric tổng hợp. Document 27936 đứng đầu với điểm 0,5058; `transformer` đóng góp 0,4600 và `model` đóng góp 0,0459, nhưng `transformer` ở đây là biến áp điện trên xe caravan. Document 4289 cũng dùng `transformer` theo nghĩa thiết bị điện, còn các document 25428, 24482 và 701 chỉ khớp `language` trong ngữ cảnh giao diện hoặc học ngoại ngữ.

### 12.3. Failure case quan trọng nhất

Query `transformer language model` cho thấy một từ hiếm có thể chi phối cosine similarity dù mang sai nghĩa. Dữ liệu đóng góp chỉ ra `transformer` tạo phần lớn điểm của document 27936, trong khi toàn văn xác nhận đây không phải kiến trúc Transformer trong NLP.

## 13. Part J — From Failure to the Next NLP Representation

Một hướng cải thiện là dùng biểu diễn ngữ nghĩa hoặc biểu diễn có ngữ cảnh. Khi đó, các cụm có nghĩa gần nhau có thể được đặt gần nhau hơn trong không gian vector. Cách này cũng có thể giúp phân biệt `transformer` là biến áp điện với `Transformer` trong xử lý ngôn ngữ.

## 14. AI Usage Policy

Việc sử dụng công cụ hỗ trợ được khai báo trong `README.md` và `reflection.md`.

## 15. Learning Check

1. Ma trận TF-IDF thưa vì mỗi tài liệu chỉ chứa một phần rất nhỏ vocabulary của toàn bộ corpus.

2. Từ xuất hiện trong nhiều tài liệu có DF lớn nên IDF thấp.

3. IDF cao chưa đủ để TF-IDF cao. Từ đó vẫn phải xuất hiện trong tài liệu thì mới có trọng số TF-IDF.

4. Cosine similarity so sánh hướng của hai vector nên ít bị ảnh hưởng bởi độ dài tài liệu.

5. Preprocessing thay đổi các token, vocabulary, TF, DF và IDF. Vì vậy nó có thể làm thay đổi điểm similarity và thứ tự kết quả tìm kiếm.

6. Với query `transformer language model`, tài liệu về biến áp điện được xếp cao vì có từ `transformer`, nhưng nội dung không đúng chủ đề.

7. Lỗi này cho thấy cần có cách biểu diễn hiểu được nghĩa và ngữ cảnh, thay vì chỉ dựa vào các từ xuất hiện giống nhau.